# AMR-UnderDifferentNoises-DL: Fine-Tuning AWGN Models on Faded Datasets
Bu notebook, önceden AWGN verisiyle eğitilmiş olan **MCLDNN** ve **PETCGDNN** modellerini, Drive üzerindeki **Rayleigh**, **Rician K3** ve **Rician K10** veri setlerinde (transfer learning / fine-tuning) eğitmeyi sağlar.

**Adımlar:**
1. Google Drive bağlanır.
2. Proje dizinindeki `amr_all_in_one_core_toolkit.py` yüklenir.
3. Modeller düşük bir *learning rate* ile tekrar eğitilir (fine-tuning).
4. Sonuçlar Drive üzerinde yeni bir `fine_tuning_results` klasörüne kaydedilir, böylece eski verileriniz korunur.

In [ ]:
# 1. Google Drive'ı Bağla
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_DIR = '/content/drive/MyDrive/AMR-Project'
RESULTS_DIR = '/content/drive/MyDrive/AMR-Project/results' # Okuma yapılacak klasör
FINETUNE_RESULTS_DIR = '/content/drive/MyDrive/AMR-Project/fine_tuning_results' # Yeni kayıt klasörü

if not os.path.exists(PROJECT_DIR):
    print(f"Uyarı: {PROJECT_DIR} bulunamadı. Lütfen dizin yolunu doğru ayarlayın.")
else:
    sys.path.append(PROJECT_DIR)
    print("Proje dizini eklendi.")

# Hata ayıklama: Drive'ın gerçekten okunup okunamadığını kontrol edelim
if os.path.exists(RESULTS_DIR):
    print("\nRESULTS_DIR içindeki mevcut klasörler:")
    print(os.listdir(RESULTS_DIR))
else:
    print(f"\nUyarı: {RESULTS_DIR} klasörü bulunamadı!")

# Toolkit yükleniyor
try:
    from amr_all_in_one_core_toolkit import *
    print("amr_all_in_one_core_toolkit modülü başarıyla yüklendi.")
except ImportError:
    print("HATA: amr_all_in_one_core_toolkit.py bulunamadı. PROJECT_DIR yolunda olduğundan emin olun.")


In [ ]:
# 2. Gerekli Kütüphaneler ve Ayarlar
import numpy as np
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Fine-tuning parametreleri
BATCH_SIZE = 400
EPOCHS = 30         # Fine-tuning genelde az epoch ile yapılır
LEARNING_RATE = 1e-4 # Düşük learning rate (örn. 0.0001)

# Veri setlerinin yolları
DATASETS = {
    'rayleigh': os.path.join(PROJECT_DIR, 'data', 'RML2016.10a_rayleigh.pkl'),
    'rician_K3': os.path.join(PROJECT_DIR, 'data', 'RML2016.10a_rician_k3.pkl'),
    'rician_K10': os.path.join(PROJECT_DIR, 'data', 'RML2016.10a_rician_k10.pkl')
}

# Hazır AWGN Modellerinin Yolları (Eski modeller RESULTS_DIR içinden okunuyor)
PRETRAINED_MODELS = {
    'MCLDNN': os.path.join(RESULTS_DIR, 'mcldnn_awgn', 'best_weights.h5'),
    'PETCGDNN': os.path.join(RESULTS_DIR, 'petcgdnn_awgn', 'best_weights.keras')
}


In [ ]:
# 3. Fine-Tune Fonksiyonu
def run_fine_tuning(model_name, dataset_name, dataset_path, pretrained_weights):
    print(f"\n{'='*60}")
    print(f"Starting Fine-Tuning: {model_name} on {dataset_name}")
    print(f"{'='*60}")
    
    # 1. Veri Yükleme
    print("Veri yükleniyor...")
    (mods, snrs, lbl), (X_train, Y_train), (X_val, Y_val), (X_test, Y_test), (train_idx, val_idx, test_idx) = load_data(dataset_path)
    
    # 2. Modeli Hazırlama
    if model_name.upper() == 'MCLDNN':
        data = prepare_data_mcldnn(X_train, X_val, X_test)
        model = MCLDNN(weights=None, classes=len(mods))
    elif model_name.upper() == 'PETCGDNN':
        data = prepare_data_petcgdnn(X_train, X_val, X_test)
        model = PETCGDNN(weights=None, classes=len(mods))
    else:
        raise ValueError("Bilinmeyen model")
        
    # 3. AWGN Ağırlıklarını Yükleme
    print(f"Önceden eğitilmiş AWGN ağırlıkları yükleniyor:\n{pretrained_weights}")
    model.load_weights(pretrained_weights)
    
    model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=LEARNING_RATE), metrics=['accuracy'])
    
    # 4. Kayıt Klasörü ve Callback'ler (YENİ KLASÖRE KAYIT YAPILACAK)
    save_dir = os.path.join(FINETUNE_RESULTS_DIR, f'{model_name.lower()}_{dataset_name.lower()}')
    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, "best_weights.weights.h5")
    
    callbacks = [
        ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_best_only=True, save_weights_only=True, mode='auto'),
        EarlyStopping(monitor='val_loss', patience=10, verbose=1, mode='auto', restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1, min_lr=1e-6),
        CSVLogger(os.path.join(save_dir, 'finetune_log.csv'))
    ]
    
    # 5. Eğitimi Başlat
    print("\nEğitim (Fine-Tuning) başlıyor...")
    history = model.fit(
        data['train'], Y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=2,
        validation_data=(data['val'], Y_val),
        callbacks=callbacks
    )
    
    # 6. Değerlendirme
    print("\nDeğerlendirme yapılıyor...")
    evaluate_model(
        model, data['test'], Y_test, lbl, test_idx, snrs, mods,
        batch_size=BATCH_SIZE, results_dir=save_dir
    )
    
    # 7. Sonuçları Çizdir ve Kaydet
    plot_training_history(history, save_dir=save_dir)
    print(f"\n>>> {model_name} on {dataset_name} Fine-Tuning TAMAMLANDI! Sonuçlar: {save_dir}")


In [ ]:
# 4. Tüm Veri Setleri ve Modeller için Döngüyü Çalıştır
for model_name, pretrained_weights in PRETRAINED_MODELS.items():
    if not os.path.exists(pretrained_weights):
        print(f"\nHATA: {pretrained_weights} bulunamadı! Lütfen model ağırlıklarının yolunu kontrol edin.")
        continue
        
    for dataset_name, dataset_path in DATASETS.items():
        if not os.path.exists(dataset_path):
            print(f"\nHATA: {dataset_path} bulunamadı! Lütfen veri seti yolunu kontrol edin.")
            continue
            
        # Belleği temizle
        tf.keras.backend.clear_session()
        
        # Modeli fine-tune et
        run_fine_tuning(model_name, dataset_name, dataset_path, pretrained_weights)

print("\nTÜM İŞLEMLER BAŞARIYLA TAMAMLANDI!")
